# Module 3: Graph data — Graph Neural Networks

<!-- Pointer: Module goal:
     - Recognize when data is naturally a graph (molecules, protein contacts, networks)
     - Understand permutation symmetry as the defining constraint for graph data
     - Build a basic Graph Convolutional layer and see it work on MUTAG
     - Connect to inductive bias: GCN encodes permutation equivariance by construction
-->

In [ ]:
import sys
sys.path.append("./src")

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. What is graph data?

<!-- Pointer:
     - A graph is nodes + edges. Each node has features; sometimes edges do too.
     - Examples: molecules (atoms + bonds), proteins (residues + contacts),
       citation networks, road networks
     - Bridge from CNN: an image is a graph too -- pixels + grid edges -- but maximally regular
     - The defining property of *general* graphs: no canonical ordering of the nodes
     - Same molecule, different node labeling -> different feature/adjacency matrices
       but it should produce the same prediction
     - This is the central constraint a GNN must satisfy: permutation symmetry
-->

## 2. Build a graph by hand

<!-- Pointer: hands-on graph construction.
     - Walk through methanol (CH3OH): 6 atoms, 5 bonds
     - Two tensors describe a graph for ML:
         X (node feature matrix): shape [N, d]
         A (adjacency matrix):    shape [N, N]
     - Inspect node degrees, draw the graph (networkx)
     - Demonstrate the permutation problem: relabel the atoms,
       show that A and X change but the molecule is the same
-->

In [ ]:
# TODO: build methanol as a graph.
#   - Define node features (one-hot atom type)
#   - Define edges (pairs of node indices)
#   - Construct X (shape [6, 4]) and A (shape [6, 6])
#   - Visualize with networkx
# Use the explore_graph_data.py script from earlier as the template if needed.

In [ ]:
# TODO: demonstrate the permutation problem.
#   - Relabel the atoms (e.g. swap atom indices)
#   - Recompute X and A
#   - Show that the matrices look totally different but describe the same molecule

## 3. The Graph Convolutional layer

<!-- Pointer:
     - Core idea: each node's new features = aggregate of its neighbors' features
     - Math:  H = ReLU( (A + I) @ X @ W )
         (A + I) -- adjacency with self-loops, so each node sees its own features too
         X       -- node features
         W       -- learnable weight matrix (the only parameters of the layer)
     - Why it satisfies permutation equivariance:
         * The same W is applied to every node's neighborhood
         * Sum/mean over neighbors is order-independent
     - Stacking layers expands the receptive field by one hop per layer (analogous to CNN)
     - For graph-level prediction, mean-pool over nodes -> permutation invariant readout
-->

In [ ]:
class GCNLayer(nn.Module):
    """One layer of:  H = ReLU((A + I) @ X @ W)"""
    def __init__(self, in_dim, out_dim):
        super().__init__()
        # TODO: define a single nn.Linear from in_dim -> out_dim
        pass

    def forward(self, X, A_hat):
        # TODO:
        # 1. transform X with the linear layer
        # 2. left-multiply by A_hat to aggregate over neighbors
        # 3. apply ReLU
        pass


class GCN(nn.Module):
    """Two GCN layers + mean-pool readout + linear classifier."""
    def __init__(self, in_dim, hidden_dim=32, num_classes=2):
        super().__init__()
        # TODO: gcn1, gcn2, classifier
        pass

    def forward(self, X, A):
        # TODO:
        # 1. add self-loops:  A_hat = A + I
        # 2. pass through gcn1, then gcn2
        # 3. mean-pool node features -> graph-level vector
        # 4. classifier head -> logits
        pass

## 4. Verify permutation equivariance numerically

<!-- Pointer:
     - Quick sanity check that the GCN really is permutation-invariant at the graph level
     - Take methanol, push it through the (untrained) GCN, get a prediction p1
     - Permute the nodes, push the permuted graph through the same GCN, get p2
     - p1 and p2 should be (numerically) identical
     - Important: this works WITHOUT training -- it's a structural property of the architecture,
       not something the model learns
-->

In [ ]:
# TODO:
#   - Instantiate an (untrained) GCN
#   - Run on (X, A)         -> p1
#   - Run on (X_perm, A_perm) -> p2
#   - Assert torch.allclose(p1, p2)

## 5. Experiment: train on MUTAG

<!-- Pointer: real-data experiment.
     - MUTAG: 188 small molecules, predict mutagenicity (binary)
     - Provide a dataset loader (parses the TU Dortmund file format)
     - Standard train/test split, train the GCN, report test accuracy
     - For comparison: also train an FC baseline that flattens (padded X, A) into a vector
       -> shows FC struggles to handle variable-size, unordered graph inputs
     - For 40 minutes in class, keep it light: train one model, look at the result
       Save the FC-vs-GCN comparison as a take-home exercise
-->

In [ ]:
# TODO: load MUTAG.


In [ ]:
# TODO: training loop.


## 6. Evaluation and discussion

<!-- Pointer:
     - Plot loss/accuracy curves
     - Discussion questions:
       * What would happen if we used an FC that consumed the flattened (X, A)?
       * Why isn't there a notion of "position" in a graph?
       * For protein contact graphs, would 2 GCN layers be enough? (No -- diameter much larger)
     - Foreshadow: sequences are yet another data type with their own ordering structure
       -> next module, transformers
-->

In [ ]:
# TODO: loss curve plot, test accuracy reporting

## 7. Submission

<!-- Pointer:
     - Students upload their final test accuracy + loss curve plot
     - Take-home (4-week project): try this on a different TUDataset or MoleculeNet dataset
-->